In [ ]:
import math
import numpy as np
import sympy as sp
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

# Constants
PHI = (1 + math.sqrt(5)) / 2  # Golden ratio
E = math.exp(1)
DEFAULT_C = -0.00247  # From reference
DEFAULT_K_STAR = 0.04449  # From reference
ZETA_SCALE = 0.1  # Empirical scale for zeta correction

def num_divisors(n):
    """Number of divisors d(n)"""
    return len(list(sp.divisors(n)))

def delta_n(n):
    """Frame shift Δ_n = d(n) * ln(n+1) / e^2"""
    return num_divisors(n) * math.log(n + 1) / (E ** 2)

def delta_max():
    """Bounded maximum shift ≈ e^2"""
    return E ** 2

def z_shift(n):
    """Invariant Z = n * (Δ_n / Δ_max)"""
    return n * (delta_n(n) / delta_max())

def geodesic_mapping(n, k=0.3):
    """Geodesic θ'(n, k) = φ * {n/φ}^k"""
    x = (n % PHI) / PHI
    return PHI * (x ** k)

def approx_gamma_n(n):
    """Zeta zero asymptotic γ_n ≈ 2πn / ln(n)"""
    if n <= 1:
        return 0.0
    return 2 * math.pi * n / math.log(n)

def base_pnt_prime(k):
    """Higher-order PNT: k*(lnk + lnlnk -1 + (lnlnk-2)/lnk)"""
    k = np.asarray(k)
    result = np.zeros_like(k, dtype=float)
    mask = k >= 2
    ln_k = np.log(k[mask])
    ln_ln_k = np.log(ln_k)
    result[mask] = k[mask] * (ln_k + ln_ln_k - 1 + (ln_ln_k - 2) / ln_k)
    return result

def d_term(k):
    """Corrective d_term = (ln(p_pnt)/exp(4))^2"""
    k = np.asarray(k)
    p_pnt = base_pnt_prime(k)
    result = np.zeros_like(k, dtype=float)
    mask = p_pnt > 1
    result[mask] = (np.log(p_pnt[mask]) / math.exp(4)) ** 2
    return result

def e_term(k):
    """Corrective e_term = p_pnt^(-1/3)"""
    k = np.asarray(k)
    p_pnt = base_pnt_prime(k)
    result = np.full_like(k, float('inf'), dtype=float)
    mask = p_pnt != 0
    result[mask] = p_pnt[mask] ** (-1/3)
    return result

def z5d_prime(k, c=DEFAULT_C, k_star=DEFAULT_K_STAR, zeta_scale=ZETA_SCALE):
    """Calibrated Z5D predictor: p_pnt + c*d_term*p_pnt + k_star*e_term*p_pnt + zeta_correction"""
    p_pnt = base_pnt_prime(k)
    zeta_correction = zeta_scale * approx_gamma_n(k)  # Zeta-informed enhancement
    geo_correction = geodesic_mapping(k)  # Geodesic density optimization
    return p_pnt + c * d_term(k) * p_pnt + k_star * e_term(k) * p_pnt + zeta_correction + geo_correction

def calibrate_z5d(ks, trues):
    """Calibrate c and k_star via curve_fit"""
    def model(k, c, k_star):
        return z5d_prime(k, c, k_star, zeta_scale=0)  # Calibrate without zeta for base
    popt, _ = curve_fit(model, ks, trues, p0=[-0.002, 0.04])
    return popt

# Example calibration (subset)
ks_calib = np.array([1000, 10000, 100000])
trues_calib = np.array([sp.ntheory.prime(int(k)) for k in ks_calib])
c_fit, k_star_fit = calibrate_z5d(ks_calib, trues_calib)
print(f"Fitted: c={c_fit:.5f}, k_star={k_star_fit:.5f}")

# Full predictor with fitted params
def full_predictor(k):
    return z5d_prime(k, c_fit, k_star_fit)

# Benchmark
ns = [10, 100, 1000, 10000, 100000]
true_primes = [sp.ntheory.prime(int(n)) for n in ns]
base_preds = [base_pnt_prime(n)[0] for n in ns]  # Scalar
z5d_preds = [z5d_prime(n)[0] for n in ns]
full_preds = [full_predictor(n)[0] for n in ns]

# Error metrics
def rel_error(pred, true):
    return abs(pred - true) / true * 100

# Table
print("| n     | True Prime | Base PNT | Z5D Prime | Full Z | Base Rel Err (%) | Z5D Rel Err (%) | Full Rel Err (%) |")
print("|-" * 8 + "|")
for i, n in enumerate(ns):
    true = true_primes[i]
    base_err = rel_error(base_preds[i], true)
    z5d_err = rel_error(z5d_preds[i], true)
    full_err = rel_error(full_preds[i], true)
    print(f"| {n} | {true} | {base_preds[i]:.2f} | {z5d_preds[i]:.2f} | {full_preds[i]:.2f} | {base_err:.2f} | {z5d_err:.2f} | {full_err:.2f} |")

# Zeta correlation (demo with first 10 zeros for validation)
zeta_zeros_demo = [14.1347, 21.0220, 25.0109, 30.4249, 32.9351, 37.5862, 40.9187, 43.3271, 48.0052, 49.7738]
approx_zeros = [approx_gamma_n(i+1) for i in range(10)]
r, p = pearsonr(approx_zeros, zeta_zeros_demo)
print(f"Zeta Asymptotic Correlation: r = {r:.4f}, p = {p:.2e}")
